In [ ]:
# basic shape and schema

talent_df = spark.sql("SELECT * FROM talent_raw")
print((talent_df.count(), len(talent_df.columns)))
talent_df.printSchema()

In [ ]:
# preview the data

# talent_df.show(5, truncate=False)
total = talent_df.count()
distinct = talent_df.distinct().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicates: {total - distinct}")

display(talent_df.limit(5).toPandas())

In [ ]:
# null counts per column
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = talent_df.select(
    [
        spark_sum(when(col(f"`{c}`").isNull(), 1).otherwise(0)).alias(c)
        for c in talent_df.columns
    ]
)
# null_counts.show(truncate=False)

null_counts_pd = null_counts.toPandas()
display(null_counts_pd)

In [ ]:
# check for duplicate rows
total = talent_df.count()
distinct = talent_df.distinct().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicates: {total - distinct}")

In [ ]:
# categorical column distinct values
from pyspark.sql.functions import countDistinct

result = talent_df.select(
    countDistinct("result").alias("distinct_results"),
    countDistinct("course_interest").alias("distinct_courses"),
    countDistinct("geo_flex").alias("distinct_geo_flex"),
    countDistinct("self_development").alias("distinct_self_dev"),
    countDistinct("financial_support_self").alias("distinct_financial"),
)
display(result.toPandas())

In [ ]:
# categorical value breakdown
for col_name in [
    "result",
    "course_interest",
    "geo_flex",
    "self_development",
    "financial_support_self",
]:
    print(f"\n--- {col_name} ---")
    display(
        talent_df.groupBy(col_name).count().orderBy("count", ascending=False).toPandas()
    )

In [ ]:
# Date Format
# observation: date stored as string in dd/MM/yyyy format, needs casting

display(talent_df.select("date").distinct().limit(20).toPandas())

In [ ]:
# Tech Score Analysis
# observation: NULLs represent no experience, not missing data, keep as NULL in transform

tech_cols = [c for c in talent_df.columns if c.startswith("tech_self_score")]
print(f"Tech score columns: {len(tech_cols)}")
print()
for c in tech_cols:
    non_null = talent_df.filter(col(f"`{c}`").isNotNull()).count()
    print(
        f"{c}: {non_null} candidates scored this language ({round(non_null/3105*100, 1)}%)"
    )

In [ ]:
# Array Columns
# observation: strengths and weaknesses are arrays — need exploding into separate tables
# to match ERD CandidateStrength and CandidateWeakness

display(talent_df.select("name", "strengths", "weaknesses").limit(10).toPandas())

In [ ]:
# Source File
# observation: source_file contains full path e.g. Talent/12711.json
# Candidate ID can be extracted as the numeric part of the filename

display(talent_df.select("source_file").limit(10).toPandas())

### Cleaning begins here

In [ ]:
from pyspark.sql.functions import col, to_date, regexp_extract, explode, lit
import pandas as pd

# Load raw table
talent_df = spark.sql("SELECT * FROM talent_raw")
print(f"Raw talent loaded: {talent_df.count()} rows, {len(talent_df.columns)} columns")

In [ ]:
# step 1: Extract candidate_id from source_file
# Talent/12711.json → 12711
talent_df = talent_df.withColumn(
    "candidate_id", regexp_extract(col("source_file"), r"(\d+)\.json", 1).cast("int")
)

display(talent_df.select("source_file", "candidate_id").limit(5).toPandas())

In [ ]:
from pyspark.sql.functions import to_date, regexp_replace

# Fix double slashes and cast in one step
talent_df = talent_df.withColumn(
    "date", to_date(regexp_replace(col("date"), "//", "/"), "dd/MM/yyyy")
)

null_dates = talent_df.filter(col("date").isNull()).count()
print(f"Dates that failed to parse after fix: {null_dates}")

display(talent_df.select("candidate_id", "date").limit(10).toPandas())

In [ ]:
# step 3: Rename tech_self_score.* columns
rename_map = {
    "tech_self_score.C#": "tech_score_csharp",
    "tech_self_score.Java": "tech_score_java",
    "tech_self_score.R": "tech_score_r",
    "tech_self_score.JavaScript": "tech_score_javascript",
    "tech_self_score.Python": "tech_score_python",
    "tech_self_score.C++": "tech_score_cpp",
    "tech_self_score.Ruby": "tech_score_ruby",
    "tech_self_score.SPSS": "tech_score_spss",
    "tech_self_score.PHP": "tech_score_php",
}

for old_name, new_name in rename_map.items():
    talent_df = talent_df.withColumnRenamed(old_name, new_name)

print("Renamed columns:")
print([c for c in talent_df.columns if c.startswith("tech_score")])

In [ ]:
# step 4: Build core candidates DataFrame
# Drop source_file — no longer needed now we have candidate_id
candidates_df = talent_df.select(
    "candidate_id",
    "name",
    "date",
    "self_development",
    "geo_flex",
    "financial_support_self",
    "result",
    "course_interest",
    "strengths",
    "weaknesses",
)

print(f"Candidates: {candidates_df.count()} rows")
display(candidates_df.limit(5).toPandas())

In [ ]:
# step 5: Build tech scores DataFrame
from pyspark.sql.functions import stack, lit, col

tech_scores_df = talent_df.select(
    "candidate_id",
    stack(
        lit(9),
        lit("C#"),
        col("tech_score_csharp"),
        lit("Java"),
        col("tech_score_java"),
        lit("R"),
        col("tech_score_r"),
        lit("JavaScript"),
        col("tech_score_javascript"),
        lit("Python"),
        col("tech_score_python"),
        lit("C++"),
        col("tech_score_cpp"),
        lit("Ruby"),
        col("tech_score_ruby"),
        lit("SPSS"),
        col("tech_score_spss"),
        lit("PHP"),
        col("tech_score_php"),
    ).alias("language", "score"),
).filter(col("score").isNotNull())

print(f"Tech scores: {tech_scores_df.count()} rows")
display(tech_scores_df.limit(10).toPandas())

In [ ]:
# step 6 : Build Technology table
# Extract distinct language names and assign a technology_id
from pyspark.sql.functions import monotonically_increasing_id

technology_df = (
    tech_scores_df.select("language")
    .distinct()
    .withColumn("technology_id", monotonically_increasing_id().cast("int") + 1)
)

display(technology_df.toPandas())

In [ ]:
# step 7 : Build CandidateTechnology table 
# Join tech scores with technology lookup to replace language name with technology_id
candidate_technology_df = tech_scores_df.join(
    technology_df, on="language", how="left"
).select("candidate_id", "technology_id", "score")

print(f"CandidateTechnology: {candidate_technology_df.count()} rows")
display(candidate_technology_df.limit(10).toPandas())

In [ ]:
# step 6: Explode strengths into separate rows
# strengths_df = talent_df.select(
#     "candidate_id", explode(col("strengths")).alias("strength")
# )

# print(f"Strengths: {strengths_df.count()} rows")
# display(strengths_df.limit(10).toPandas())

In [ ]:
# step 7: Explode weaknesses into separate rows
# weaknesses_df = talent_df.select(
#     "candidate_id", explode(col("weaknesses")).alias("weakness")
# )

# print(f"Weaknesses: {weaknesses_df.count()} rows")
# display(weaknesses_df.limit(10).toPandas())

In [ ]:
# step 8: Save transformed tables to Databricks
candidates_df.write.mode("overwrite").saveAsTable("candidates_clean")
candidate_technology_df.write.mode("overwrite").saveAsTable(
    "candidate_technology_clean"
)
technology_df.write.mode("overwrite").saveAsTable("technology_clean")
# strengths_df.write.mode("overwrite").saveAsTable("strengths_clean")
# weaknesses_df.write.mode("overwrite").saveAsTable("weaknesses_clean")

print("All talent transformed tables saved:")
print(" - candidates_clean")
print(" - candidate_technology_clean")
print(" - technology_clean")
# print(" - strengths_clean")
# print(" - weaknesses_clean")